# The following explores the need and motivation for our project

In [ ]:
import pandas as pd
import numpy as np
import itertools
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import importlib
import my_lstm
importlib.reload(my_lstm)

from my_lstm import build_lstm_model, create_sequences, expanding_window_lstm_forecast2

In [ ]:
# load df
df = pd.read_csv('../data/final_df.csv')

In [ ]:
# orignal real data split into 70/15/15
train_size = int(len(df) * 0.7)
val_size = int(len(df) * 0.15)
val_end = train_size + val_size


train_data = df.iloc[:train_size].copy()
val_data = df.iloc[train_size:val_end].copy()
test_data = df.iloc[val_end:].copy()

# print length of data in each set
print(f'Train data length: {len(train_data)}')
print(f'Validation data length: {len(val_data)}')
print(f'Test data length: {len(test_data)}')

# Reduce training set by 25%

In [ ]:
# drop first 25% of original training period
train_start_idx2 = int(len(train_data) * 0.25)

train_df2 = train_data.iloc[train_start_idx2:].copy()
print("Reduced training length:", len(train_df2))
train_df2.head()

In [ ]:
# grid search over hyperparameters for the expanding window LSTM
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid = []

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    try:
        val_forecasts = expanding_window_lstm_forecast2(
            df=df,
            feature_cols=['wti_ret'],
            target_col='wti_ret',
            date_col="Date",
            train_start_idx=train_start_idx2,   
            initial_train_size=train_size,      # keep original validation start
            end_idx=val_end,                    # keep original validation end
            lookback=lb,
            units=units,
            dropout=dr,
            epochs=ep,
            batch_size=32,
            verbose=0,
            scale=True,
            seed=42
        )

        if len(val_forecasts) == 0:
            print("No forecasts generated, skipping.")
            continue

        mse = mean_squared_error(
            val_forecasts["actual"],
            val_forecasts["predicted"]
        )

        print(f"Validation MSE: {mse:.6f}")

        results_grid.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep,
            "mse": mse
        })

    except Exception as e:
        print(f"Error: {e}")
        continue

results_grid = pd.DataFrame(results_grid).sort_values("mse").reset_index(drop=True)

In [ ]:
results_df = pd.DataFrame(results_grid)
results_df = results_df.sort_values("mse")

print(results_df.head())

best_params = results_df.iloc[0]
print(best_params)

## Out of sample test

In [ ]:
test_results = expanding_window_lstm_forecast2(
    df=df,
    feature_cols=['wti_ret'],
    target_col='wti_ret',
    train_start_idx=train_start_idx2,  
    initial_train_size=val_end,         # original test start
    end_idx=len(df),                    # forecast through end of sample
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)

In [ ]:
# export test results to csv
test_results.to_csv('results/lstm_reduce25_results.csv', index=False)

# Reduce training data by 50%

In [ ]:
# drop first 50% of original training period
train_start_idx3 = int(len(train_data) * 0.50)

train_df3 = train_data.iloc[train_start_idx3:].copy()
print("Reduced training length:", len(train_df3))
train_df3.head()


In [ ]:
# grid search over hyperparameters for the expanding window LSTM
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid3 = []

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    try:
        val_forecasts = expanding_window_lstm_forecast2(
            df=df,
            feature_cols=['wti_ret'],
            target_col='wti_ret',
            date_col="Date",
            train_start_idx=train_start_idx3,   
            initial_train_size=train_size,      # keep original validation start
            end_idx=val_end,                    # keep original validation end
            lookback=lb,
            units=units,
            dropout=dr,
            epochs=ep,
            batch_size=32,
            verbose=0,
            scale=True,
            seed=42
        )

        if len(val_forecasts) == 0:
            print("No forecasts generated, skipping.")
            continue

        mse = mean_squared_error(
            val_forecasts["actual"],
            val_forecasts["predicted"]
        )

        print(f"Validation MSE: {mse:.6f}")

        results_grid.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep,
            "mse": mse
        })

    except Exception as e:
        print(f"Error: {e}")
        continue

results_grid3 = pd.DataFrame(results_grid3).sort_values("mse").reset_index(drop=True)

In [ ]:
results_df3 = pd.DataFrame(results_grid3)
results_df3 = results_df3.sort_values("mse")

print(results_df3.head())

best_params = results_df3.iloc[0]
print(best_params)

### Out of sample test

In [ ]:
test_results = expanding_window_lstm_forecast2(
    df=df,
    feature_cols=['wti_ret'],
    target_col='wti_ret',
    train_start_idx=train_start_idx3,  
    initial_train_size=val_end,         # original test start
    end_idx=len(df),                    # forecast through end of sample
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)

In [ ]:
# export test results to csv
test_results.to_csv('results/lstm_reduce50_results.csv', index=False)

# Reduce data by 75%

In [ ]:
# drop first 75% of original training period
train_start_idx4 = int(len(train_data) * 0.50)

train_df4 = train_data.iloc[train_start_idx4:].copy()
print("Reduced training length:", len(train_df4))
train_df4.head()

In [ ]:
# grid search over hyperparameters for the expanding window LSTM
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid4 = []

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    try:
        val_forecasts = expanding_window_lstm_forecast2(
            df=df,
            feature_cols=['wti_ret'],
            target_col='wti_ret',
            date_col="Date",
            train_start_idx=train_start_idx4,   
            initial_train_size=train_size,      # keep original validation start
            end_idx=val_end,                    # keep original validation end
            lookback=lb,
            units=units,
            dropout=dr,
            epochs=ep,
            batch_size=32,
            verbose=0,
            scale=True,
            seed=42
        )

        if len(val_forecasts) == 0:
            print("No forecasts generated, skipping.")
            continue

        mse = mean_squared_error(
            val_forecasts["actual"],
            val_forecasts["predicted"]
        )

        print(f"Validation MSE: {mse:.6f}")

        results_grid.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep,
            "mse": mse
        })

    except Exception as e:
        print(f"Error: {e}")
        continue

results_grid4 = pd.DataFrame(results_grid4).sort_values("mse").reset_index(drop=True)

In [ ]:
results_df4 = pd.DataFrame(results_grid4)
results_df4 = results_df4.sort_values("mse")

print(results_df4.head())

best_params = results_df4.iloc[0]
print(best_params)

# Out of sample test

In [ ]:
test_results = expanding_window_lstm_forecast2(
    df=df,
    feature_cols=['wti_ret'],
    target_col='wti_ret',
    train_start_idx=train_start_idx4,  
    initial_train_size=val_end,         # original test start
    end_idx=len(df),                    # forecast through end of sample
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)

In [ ]:
# export test results to csv
test_results.to_csv('results/lstm_reduce75_results.csv', index=False)